# Bài 29 — chuỗi luyện Easy · LeetCode #144 — Binary Tree Preorder Traversal

**Mức độ:** Easy · **Ngôn ngữ:** Python 3 · **Đề gốc:** https://leetcode.com/problems/binary-tree-preorder-traversal/

**Mục tiêu học tập**

- Đọc đề và tự phân biệt input/output, constraints và ví dụ biên.
- Nắm cú pháp Python theo từng cell thực nghiệm trước khi ghép thuật toán.
- Hiểu brute force/baseline trước, sau đó giải thích mấu chốt tối ưu bằng bất biến.
- Trace từng bước với dữ liệu dễ sai, không xem code như một hộp đen.
- Chạy lời giải chuẩn LeetCode, test mẫu, edge case, randomized/exhaustive khi phù hợp.
- Tự giải **6 bài luyện từ dễ đến khó**, chỉ xem gợi ý/đáp án cuối sau khi thử.

**Cách học:** đọc Markdown → tự dự đoán output → chạy cell Python ngay sau đó → giải thích lại bằng lời. Các cell TODO không chạy tự động và chỉ phục vụ tự luyện.

## 1. Đọc đề và xác định đúng đầu vào/đầu ra

Trả danh sách các giá trị khi duyệt **Root → Left → Right**. Ở node hiện tại, ghi giá trị **trước khi** xử lý cây con.

### Ví dụ quan trọng

Cây `[1,None,2,3]` trả `[1,2,3]`; cây `[4,2,6,1,3,5,7]` trả `[4,2,1,3,6,5,7]`.

Hãy tự trả lời: **đầu ra là số, boolean, danh sách, hay node?** Với bài cấu trúc dữ liệu, phải chú ý **node/tham chiếu** khác **giá trị số**.

## 2. Thiết lập môi trường để chạy notebook

Trong Jupyter, hãy chạy cell tạo lớp dữ liệu và hàm hỗ trợ trước các cell sau. **Lưu ý:** LeetCode có sẵn một số lớp hoặc API; khi submit chỉ sao chép riêng cell `class Solution`/lớp thiết kế, không sao chép hàm giả lập dùng cho notebook.

In [ ]:
from __future__ import annotations
from typing import Optional
from collections import deque

class TreeNode:
    def __init__(self, val: int = 0, left=None, right=None):
        self.val, self.left, self.right = val, left, right
    def __repr__(self):
        return f"TreeNode({self.val})"

def tree(values):
    """Tạo cây từ danh sách level-order; None = ô trống."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0]); queue=deque([root]); pos=1
    while queue and pos < len(values):
        parent=queue.popleft()
        if pos < len(values):
            v=values[pos]; pos+=1
            if v is not None:
                parent.left=TreeNode(v); queue.append(parent.left)
        if pos < len(values):
            v=values[pos]; pos+=1
            if v is not None:
                parent.right=TreeNode(v); queue.append(parent.right)
    return root

def show_tree(root):
    """Đọc cấu trúc cây bằng BFS, không in quá nhiều None cuối."""
    if root is None:
        return []
    q=deque([root]); result=[]
    while q:
        p=q.popleft()
        if p is None:
            result.append(None)
        else:
            result.append(p.val); q.append(p.left); q.append(p.right)
    while result and result[-1] is None:
        result.pop()
    return result

### 2.1. TreeNode là đối tượng, không phải phần tử của mảng

`TreeNode` là một **đối tượng** (*object*) với ba thuộc tính: `.val`, `.left` và `.right`.
Hai thuộc tính con lưu **tham chiếu** tới nút khác hoặc `None` khi không có con.
Gốc là nút duy nhất không có cha. **Lá** là nút không có cả con trái lẫn con phải.
Ví dụ gốc 8 có trái 3 và phải 10. Chúng ta đọc `root.left.val` để lấy 3,
không phải `root[1]` như truy cập list. LeetCode đã cấp sẵn cây cho lời giải;
chỉ trong notebook mới cần hàm `tree([8,3,10])` để dựng dữ liệu thử.
Đọc đoạn code và nói vì sao hai nút 3, 10 là hai đối tượng độc lập.

In [ ]:
example_root = TreeNode(8)
example_root.left = TreeNode(3)
example_root.right = TreeNode(10)
print('root:', example_root.val, 'left:', example_root.left.val, 'right:', example_root.right.val)
print('root.left is root.right?', example_root.left is example_root.right)
assert example_root.left.val == 3 and example_root.right.val == 10

### 2.2. `None`, `is None` và short-circuit tránh lỗi

Nếu `node is None`, biến đó **không trỏ đến nút thật**. Khi ấy `node.val`
sẽ gây `AttributeError`; một thuật toán cây phải kiểm tra tồn tại trước.
Biểu thức `node is not None and node.left is not None` dùng **short-circuit**:
vế sau chỉ được đánh giá nếu vế trước đúng. Đây là an toàn dữ liệu, không phải thủ thuật.
`is` kiểm tra cùng đối tượng, khác `==` thường dùng so sánh giá trị.
Với bài duyệt cây, hãy phân biệt *nút trống* với *nút có giá trị 0*:
`TreeNode(0)` là một nút thật, không được bỏ qua chỉ vì `.val == 0`.

In [ ]:
absent = None
real_zero = TreeNode(0)
for candidate in (absent, real_zero, TreeNode(-5)):
    exists = candidate is not None
    print('node=',candidate,'tồn tại?',exists,'val=',candidate.val if exists else 'không có')
assert real_zero is not None and absent is None

### 2.3. Đọc dữ liệu level-order và `None` trong ví dụ

LeetCode thường hiển thị cây theo *level-order*: các giá trị theo tầng, trái trước phải,
dùng `null` cho vị trí trống. Trong Python notebook ta viết `None`.
Chú ý list `[1, None, 2, 3]` **không** có nghĩa nút 3 là con phải của 2.
Khi dựng cây đúng kiểu level-order, gốc 1 có con phải 2; sau đó 2 có con trái 3.
Hãy dùng `show_tree(...)` để đọc ngược lại cây đã tạo. Đó là hàm học tập, **không nộp** LeetCode.
Đừng tự gán quan hệ cha-con theo công thức `2*i+1` khi có các ô `None` không đủ trong list.

In [ ]:
special = tree([1,None,2,3])
print('cấu trúc:',show_tree(special))
print('gốc:',special,'con phải:',special.right,'cháu bên trái:',special.right.left)
print('con trái gốc:',special.left)
assert special.right.left.val == 3 and special.left is None

### 2.4. Cú pháp `list` như stack LIFO

*Stack* (ngăn xếp) tuân theo **LIFO**: vào sau → ra trước.
Với Python, `stack.append(x)` là **đẩy** (*push*) và `stack.pop()` là **lấy/loại phần tử cuối**.
`pop()` vừa trả phần tử, vừa thay đổi chính list; khi rỗng thì gọi `pop()` sẽ gây lỗi.
Vì vậy chuẩn viết là `while stack:` rồi mới `stack.pop()`.
Trong duyệt cây, thứ tự `append` quyết định chính xác thứ tự xử lý;
mọi ví dụ nên in *stack trước pop*, *phần tử vừa pop* và *stack sau cập nhật*.

In [ ]:
plate_stack=[]
for label in ('đáy','giữa','đỉnh'):
    plate_stack.append(label)
print('stack lúc đầu:',plate_stack)
while plate_stack:
    print('pop ->',plate_stack.pop(),'còn:',plate_stack)
assert plate_stack == []

### 2.5. Vì sao phải đẩy con PHẢI trước con TRÁI?

Khi mục tiêu cần **xử lý nhánh trái trước nhánh phải**, quy tắc LIFO có vẻ ngược đời:
hãy đẩy **phải trước**, rồi đẩy **trái sau** để trái nằm trên đỉnh stack.
Đây là quy tắc thường gặp khi mô phỏng DFS bằng stack, nhưng mỗi kiểu traversal
còn quy định *thời điểm ghi giá trị gốc* khác nhau.
Ví dụ stack ban đầu `['root']`: pop root, append right, append left;
lần pop tiếp theo sẽ là left, không phải right. Dự đoán kết quả trước khi chạy.

In [ ]:
demo_stack=['root']
current=demo_stack.pop()
demo_stack.append('right')
demo_stack.append('left')
print('đã xử lý:',current,'stack còn:',demo_stack)
print('thứ tự các việc tiếp theo:',demo_stack.pop(),demo_stack.pop())
assert demo_stack == []

### 2.6. Đệ quy, base case, call stack và chiều cao cây

**Recursion (đệ quy)** là hàm gọi lại chính nó trên bài toán nhỏ hơn;
*base case* là điểm dừng (trong cây thường `node is None`).
Khi `dfs(root.left)` chưa trả về, lời gọi cha đang được giữ trong *call stack*.
Mỗi lần đi sâu thêm 1 tầng phải giữ thêm một lời gọi. Nếu `h` là số nút của
đường từ gốc đến lá dài nhất, đệ quy cần `O(h)` bộ nhớ ngăn xếp.
Cây lệch quá sâu có thể vượt **recursion limit** Python; giải pháp stack tường minh
cho phép viết DFS lặp và tránh giới hạn độ sâu gọi hàm.
Một `print` trong đệ quy chỉ minh họa; kết quả lời giải phải được trả bằng `return`.

In [ ]:
def enter_exit(n, indent=0):
    print('  '*indent+'vào', n)
    if n == 0:
        print('  '*indent+'dừng')
        return 0
    child=enter_exit(n-1,indent+1)
    print('  '*indent+'quay về',n)
    return 1+child
assert enter_exit(3)==3

### 2.7. Tham chiếu, đối tượng giống giá trị và output `list[int]`

Hai nút đều có `.val == 5` vẫn có thể là hai **nút khác nhau**;
traversal phải ghi **cả hai lần** số 5 nếu cây thật sự chứa hai nút như vậy.
`a = node` không tạo ra bản sao nút; cả hai tên cùng trỏ đối tượng.
Tuy nhiên `result.append(node.val)` ghi **giá trị số** vào list output, không ghi node.
Khi LeetCode yêu cầu `List[int]`, trả `[5,5]` là hợp lệ trong ví dụ có hai nút cùng giá trị.
Đừng tự dùng `set` khử trùng vì làm mất các lần xuất hiện có ý nghĩa.

In [ ]:
left_five=TreeNode(5)
right_five=TreeNode(5)
twin_root=TreeNode(1,left_five,right_five)
print('giá trị bằng?', left_five.val == right_five.val)
print('cùng đối tượng?', left_five is right_five)
print('số nút thật có giá trị 5:',sum(node.val==5 for node in (left_five,right_five)))
assert left_five is not right_five

## 3. Python nền tảng — công cụ cần hiểu trước khi giải

Stack **LIFO**: `.append(node)` thêm đỉnh, `.pop()` lấy đỉnh cuối. Vì cần xử lý trái trước, hãy **push phải trước, push trái sau**.

**Bài thí nghiệm:** thử thay dữ liệu trong cell dưới; các lệnh `print()` dùng để thấy giá trị trung gian, không phải mã submit.

In [ ]:
stack=['root'];stack.append('right');stack.append('left')
print('stack:',stack)
print('pop:',stack.pop(),'rồi',stack.pop())

### Câu hỏi gợi mở trước khi tối ưu

Sau khi tự chạy ví dụ nền tảng, hãy tự trả lời bằng lời:

1. Trong bài **Binary Tree Preorder Traversal**, đâu là dữ liệu thật, đâu là công cụ dựng thử?
2. Biến nào thay đổi sau một vòng lặp / sau một lời gọi đệ quy?
3. Khi nào một ký tự / node / giá trị được xem là 'đã xử lý xong'?
4. Nếu input rỗng, có một phần tử, hay chỉ có một nhánh, cần return kiểu gì?
5. Thao tác nào tăng theo số phần tử n, thao tác nào chỉ giữ vài biến?

Không cần trả lời bằng công thức ngay; thử input nhỏ rồi diễn đạt kết quả.

### Quy trình chạy notebook không phụ thuộc trạng thái ngầm

Các hàm dựng input, baseline, `solve`, trace và đáp án được định nghĩa theo thứ tự từ trên xuống dưới.
Không chạy phần đáp án trước phần bài tập nếu muốn tự luyện.
Khi thử một biến thể, hãy tạo input mới, không sửa input đã được dùng cho `assert` ở các cell phía trên.
Nếu chạy lại một cell, bạn nên kiểm tra các biến demo bị ghi đè; cách ổn định nhất là **Restart Kernel & Run All**.

Ở các bài Linked List/Tree, LeetCode đã cung cấp sẵn lớp node. Trong notebook, lớp node được định nghĩa để mô phỏng local và không phải mã cần nộp.

### 3.1. Ba thứ tự traversal nhìn trên cùng một cây

*Traversal* (duyệt cây) nghĩa là thăm các nút theo quy tắc cố định.
Trên cây có gốc 4, con trái 2 (với con 1,3), con phải 6 (với con 5,7):
**preorder** Root–Left–Right là `[4,2,1,3,6,5,7]`;
**inorder** Left–Root–Right là `[1,2,3,4,5,6,7]`;
**postorder** Left–Right–Root là `[1,3,2,5,7,6,4]`.
Bài #144 yêu cầu duy nhất **Root → Left → Right**. Đừng đổi thứ tự theo vị trí list input.
Có thể dùng cùng cấu trúc cây để nhìn sự khác biệt **thời điểm ghi `.val`**.
Ba thứ tự đều phải ghi đủ số nút thật, kể cả những nút trùng giá trị.

In [ ]:
chosen_tree=tree([4,2,6,1,3,5,7])
preorder_sample=[4,2,1,3,6,5,7]
inorder_sample=[1,2,3,4,5,6,7]
postorder_sample=[1,3,2,5,7,6,4]
print('cây:',show_tree(chosen_tree))
print('pre:',preorder_sample,'in:',inorder_sample,'post:',postorder_sample)
assert len(preorder_sample)==len(postorder_sample)==len(inorder_sample)==7

### 3.2. Vì sao DFS khác với duyệt level-order BFS?

**DFS (Depth-First Search)** đi sâu một nhánh trước khi quay sang nhánh khác;
preorder/postorder là các **thứ tự ghi kết quả khi DFS**.
**BFS (Breadth-First Search)** đi từng tầng, thường dùng queue FIFO.
Với cây `[1,2,3,4,5]`, BFS là `[1,2,3,4,5]`,
trong khi preorder và postorder thường khác hẳn.
Hiểu sự khác nhau này giúp tránh nhầm bài yêu cầu traversal bằng
thuật toán chỉ in theo hàng chờ. Nếu đề hỏi 'level order', mới dùng BFS.
`deque.popleft()` lấy phần tử đầu của queue, còn `list.pop()` lấy cuối của stack.

In [ ]:
from collections import deque
def level_order_demo(root):
    if root is None:return []
    q=deque([root]);values=[]
    while q:
        node=q.popleft();values.append(node.val)
        if node.left is not None:q.append(node.left)
        if node.right is not None:q.append(node.right)
    return values
small_tree=tree([1,2,3,4,5])
print('BFS:',level_order_demo(small_tree),'DFS bài đang học:',baseline(small_tree) if 'baseline' in globals() else '(sẽ học sau)')
assert level_order_demo(small_tree)==[1,2,3,4,5]

## 4. Cách nghĩ trực giác (baseline)

Đệ quy bám sát định nghĩa: ghi node trước, rồi gọi hàm cho cây trái và cây phải. Dễ hiểu nhưng stack đệ quy có thể sâu.

**Hãy dự đoán** kết quả của hàm baseline trên các ví dụ trước khi chạy.

In [ ]:
def baseline(root):
    ans=[]
    def dfs(node):
        if node is None:return
        ans.append(node.val)
        dfs(node.left);dfs(node.right)
    dfs(root);return ans

In [ ]:
for a in [[],[1,None,2,3],[4,2,6,1,3,5,7]]:print(a,'->',baseline(tree(a)))

### 4.1. Đệ quy giúp diễn đạt thứ tự duyệt bằng chính câu tiếng Việt

Cách baseline dùng `dfs(node)` nhận một cây con:
nếu node rỗng thì không làm gì; nếu có node, gọi trên trái/phải
và ghi giá trị vào `result` đúng thời điểm yêu cầu.
Với #144, thứ tự phải là **Root → Left → Right**. Mỗi nút được ghi đúng một lần.
Tư duy recursion giúp người mới nhìn bài toán **cây con cùng dạng với cây lớn**.
Nhưng call stack có thể sâu O(h); code giải chính dùng stack tường minh
để tránh rủi ro `RecursionError` trên cây dạng dây rất sâu.
Hãy kiểm cùng ví dụ với một nút, một nhánh và cây cân đối.

In [ ]:
for arr in ([],[8],[8,3,None,2],[4,2,6,1,3,5,7]):
    root=tree(arr)
    print('cây:',arr,'baseline:',baseline(root))
    assert len(baseline(root))==sum(x is not None for x in arr)

### 4.2. Recursion trông gọn nhưng bộ nhớ ngăn xếp không miễn phí

Với cây một nhánh sâu h, `dfs` gọi thêm một lần cho mỗi node.
Các lời gọi cha phải chờ con trả về nên Python giữ tới O(h) stack frames.
Nếu h đủ lớn, Python có thể báo lỗi `maximum recursion depth exceeded`.
Điều này không làm thuật toán đệ quy sai, nhưng khiến nó kém bền vững
khi input vượt recursion limit của môi trường thực thi.
Dùng explicit stack đưa trạng thái vào `list`, vẫn có O(h) extra memory,
nhưng không phụ thuộc giới hạn call stack Python. Tránh tăng recursionlimit tùy tiện.

In [ ]:
def make_left_chain_demo(n):
    root=None
    for value in range(n-1,-1,-1):
        root=TreeNode(value,root,None)
    return root
chain=make_left_chain_demo(12)
print('cây dây n=12, thứ tự duyệt:',baseline(chain))
assert len(baseline(chain))==12

## 5. Vì sao cần chuyển sang kỹ thuật chính?

Đệ quy dùng call stack ngầm. Muốn nhìn rõ trạng thái, ta dùng **stack tường minh** và cho node gốc vào stack. Pop node nào, ghi ngay node đó.

Một thuật toán tốt cần trả lời: **ta đang nhớ thông tin gì, cập nhật khi nào, và làm sao không bỏ sót trường hợp nào?**

### Bước đột phá — LIFO có thể mô phỏng chính xác Root–Left–Right

Nếu dùng đệ quy, lời gọi xử lý gốc rồi mới đi nhánh trái rồi nhánh phải.
Với stack tường minh, ta *append gốc ngay khi pop*, rồi push **con phải trước**
để con trái nằm trên cùng và được xử lý trước theo LIFO.

**Bất biến của stack:** trước mỗi lượt, stack (đọc từ đỉnh xuống)
liệt kê những gốc của các cây con cần thăm tiếp theo đúng thứ tự preorder.
Mỗi lần pop hoàn thành công việc ghi một node, và đưa đúng hai cây con
vào vị trí ưu tiên; không bỏ sót, không ghi trùng.

Sau khi stack rỗng, không còn cây con chưa thăm ⇒ output là preorder của toàn cây.

## 6. Lời giải chính — hiểu từng bước rồi mới viết

Lặp đến khi stack rỗng: lấy node ra → `ans.append(node.val)`; push con phải trước, trái sau. Stack LIFO sẽ đưa trái lên trước để xử lý.

Dưới đây là **hàm giải độc lập**. Khi submit, ta đưa cùng logic vào đúng chữ ký LeetCode.

### 5.1. Chuyển lời gọi đệ quy thành công việc nằm trên stack

Đệ quy giấu một ngăn xếp lời gọi; lời giải lặp **đặt ngăn xếp ra trước mắt**.
Với bài #144, muốn đạt thứ tự **Root → Left → Right**, ta cần lên kế hoạch chính xác
*khi nào ghi node* và *khi nào xử lý hai cây con*.
Việc thăm node không đồng nghĩa luôn phải ghi node ngay: preorder ghi sớm,
postorder ghi sau khi con hoàn thành.
**Bất biến** nên mô tả stack đang chứa những công việc nào chưa làm;
mỗi pop giải quyết một công việc hoặc tạo thêm các công việc con.
Để đúng, mọi nút thật phải được ghi đúng một lần và không bỏ sót con nào.

In [ ]:
task_stack=['root']
print('công việc ban đầu:',task_stack)
task=task_stack.pop()
print('đã lấy:',task)
task_stack.extend(['right','left'])
print('hai việc đang chờ:',task_stack,'vì pop cuối trước')
assert task_stack.pop()=='left' and task_stack.pop()=='right'

In [ ]:
def solve(root: Optional[TreeNode]) -> list[int]:
    if root is None:return []
    stack=[root];ans=[]
    while stack:
        node=stack.pop()
        ans.append(node.val)
        if node.right is not None:stack.append(node.right)
        if node.left is not None:stack.append(node.left)
    return ans

### 6.1. Preorder phải ghi gốc NGAY khi pop lần đầu

**Preorder** theo thứ tự Root → Left → Right.
Mỗi lần `node = stack.pop()`, nút đó chính là công việc tiếp theo:
lập tức `result.append(node.val)` rồi đẩy con phải (nếu có), con trái (nếu có).
LIFO khiến nhánh trái được pop trước nhánh phải.
Không cần cờ `visited` vì node được xử lý ngay khi đến lượt.
Với cây `[1,None,2,3]`, kết quả là `[1,2,3]`, **không** phải `[3,2,1]`.
Nếu bạn chỉ ghi khi đã xử lý con, đó là postorder, sai đề #144.

In [ ]:
root=tree([1,None,2,3])
stack=[root];answer=[]
while stack:
    node=stack.pop();answer.append(node.val)
    if node.right is not None:stack.append(node.right)
    if node.left is not None:stack.append(node.left)
print('preorder:',answer)
assert answer==[1,2,3]

### 6.2. Vì sao push con phải rồi con trái luôn cho trái trước?

Ta giả sử khi đang xử lý root, cả hai con tồn tại.
Nếu `stack.append(root.right)` rồi `stack.append(root.left)`, phần tử trên cùng
là con trái; vì `pop()` lấy phần tử **cuối**, con trái được xử lý trước.
Cấu trúc con phải được giữ phía dưới chờ cho cả nhánh trái xử lý xong.
Đây là cách stack mô phỏng quy tắc đệ quy `dfs(left); dfs(right)`.
Nếu push trái rồi phải, output của cây cân đối sẽ thành Root–Right–Left,
sai ở ví dụ có ít nhất hai con.

In [ ]:
r=TreeNode(8,TreeNode(3),TreeNode(10))
todo=[r]
got=todo.pop()
todo.append(got.right)
todo.append(got.left)
print('top stack:',todo[-1],'sẽ pop trước:',todo.pop())
assert todo.pop().val==10

## 7. Trace bằng tay — quan sát trạng thái, không chỉ đáp án

| Lượt | Stack trước (đỉnh phải) | Pop | ans sau |
|---:|---|---|---|
| 1 | `[4]` | 4 | `[4]` |
| 2 | `[6,2]` | 2 | `[4,2]` |
| 3 | `[6,3,1]` | 1 | `[4,2,1]` |
| 4 | `[6,3]` | 3 | `[4,2,1,3]` |

Đọc từ trái sang phải: mỗi dòng ghi **trạng thái trước khi quyết định**, hành động và phần thay đổi **sau khi quyết định**.

In [ ]:
def trace_preorder(root):
    if root is None:return []
    st=[root];ans=[]
    while st:
        print('stack trước:',[x.val for x in st])
        node=st.pop();ans.append(node.val)
        if node.right:st.append(node.right)
        if node.left:st.append(node.left)
        print('pop',node.val,'-> ans',ans)
    return ans
trace_preorder(tree([4,2,6,1,3,5,7]))

### 7.1. Trace chi tiết preorder: stack trước, pop, output sau, push con

Dùng cây `[4,2,6,1,3,5,7]` để nhìn rõ cả hai nhánh.
Với mỗi vòng, ta in **stack trước**, node vừa pop, `answer` sau khi ghi,
rồi các con được push. Cần giữ quy ước **đỉnh stack ở bên phải** danh sách.
Root 4 pop trước, sau đó 2,1,3; nhánh 6 chỉ được xử lý khi nhánh trái hết.
Đây là bằng chứng trực quan rằng push right trước left phục vụ thứ tự LIFO.
Tự dựng 3 dòng đầu vào giấy trước khi chạy.

In [ ]:
def trace_preorder_full144(root):
    stack=[root] if root else [];out=[];turn=0
    while stack:
        turn+=1
        print('trước',turn,':',[x.val for x in stack])
        node=stack.pop();out.append(node.val)
        print('  pop',node.val,'-> out',out)
        if node.right:stack.append(node.right)
        if node.left:stack.append(node.left)
        print('  stack sau:',[x.val for x in stack])
    return out
assert trace_preorder_full144(tree([4,2,6,1,3,5,7]))==[4,2,1,3,6,5,7]

### 7.2. Cây chỉ có một nhánh: không được bỏ sót node

Khi mỗi node chỉ có **con phải**, thuật toán không được giả định `node.left` luôn tồn tại.
Với chuỗi 1→2→3 theo nhánh phải, preorder là `[1,2,3]`.
Cây chỉ có **con trái** cũng có cùng thứ tự theo chuỗi đi xuống, vì mỗi node
chỉ có một đường tiếp theo. So sánh cây rỗng, một nút, nhánh trái, nhánh phải.
`if node.left is not None:` và `if node.right is not None:`
là cách kiểm biên rõ nghĩa, không đọc `.val` khi None.

In [ ]:
for arr in ([],[1],[1,2,None,3],[1,None,2,None,3]):
    result=solve(tree(arr))
    print('cây:',arr,'->',result)
    assert result==baseline(tree(arr))

### 7.3. Thứ tự ghi giá trị quyết định tên thuật toán

Cùng một cây, preorder ghi Root **trước** hai cây con,
inorder ghi Root **giữa** trái và phải, postorder ghi Root **sau** cả hai.
Khác biệt không nằm ở việc những node nào được tới thăm,
mà nằm ở *thời điểm append `.val`*.
Với cây cân đối `[2,1,3]`, ba kết quả lần lượt là `[2,1,3]`,
`[1,2,3]`, `[1,3,2]`. Cây dạng đường một nhánh có thể vô tình cho hai thứ tự giống,
nên phải test thêm cây có hai con để tránh nhầm thuật toán.

In [ ]:
r=tree([2,1,3])
pre=solve(r)
ino=[1,2,3]
post=[1,3,2]
print('preorder:',pre,'inorder:',ino,'postorder:',post)
assert pre==[2,1,3] and pre!=ino and pre!=post

### 7.4. So recursion và explicit stack cùng logic Root–Left–Right

Đệ quy `visit(root); dfs(left); dfs(right)` và iterative stack
thực hiện cùng thứ tự dù cách lưu công việc khác nhau.
Ta có thể đếm *số lần ghi* và *số nút đã xử lý* để kiểm:
mỗi nút thật xuất hiện đúng một lần, không có nút trống trong output.
Test thêm cây có giá trị trùng (`[5,5,5]`) nhằm phát hiện lỗi dùng set
hoặc loại trùng ngoài yêu cầu.
Hai hàm phải trả cùng list số với mọi cây hợp lệ trong các ví dụ.

In [ ]:
for arr in ([5,5,5],[5,1,1,0,None,None,0],[1,None,2,3]):
    root=tree(arr)
    a=baseline(root);b=solve(root)
    print('cây:',arr,'baseline:',a,'stack:',b)
    assert a==b and len(b)==sum(v is not None for v in arr)

## 8. Vì sao thuật toán đúng? — lập luận bằng bất biến

Mỗi node được push đúng một lần bởi cha và pop đúng một lần. Khi pop node, ghi giá trị trước các con. Push con phải trước trái nên trái nằm ở đỉnh stack và được xử lý trước con phải. Do đó thứ tự thu được là Root-Left-Right.

**Hai câu cần trả lời:** (1) Sau mỗi bước, điều gì vẫn đúng? (2) Khi dừng, vì sao giá trị trả về chính là đáp án theo yêu cầu?

## 9. Code nộp LeetCode (Python 3)

**Chỉ lấy cell bên dưới để nộp.** Phần dựng input và `print()` ở các cell khác chỉ phục vụ học tập.

In [ ]:
from typing import Optional
class Solution:
    def preorderTraversal(self, root: Optional[TreeNode]) -> list[int]:
        if root is None:
            return []
        stack = [root]
        result = []
        while stack:
            node = stack.pop()
            result.append(node.val)
            if node.right is not None:
                stack.append(node.right)
            if node.left is not None:
                stack.append(node.left)
        return result

## 10. Kiểm thử đúng đầu ra, trường hợp biên và lỗi dễ bỏ sót

Các lệnh `assert` sẽ dừng với thông báo lỗi nếu kết quả **khác** giá trị mong đợi. Tập test bao gồm ví dụ dễ hiểu, input nhỏ nhất và tình huống dễ nhầm. Đừng chỉ kiểm tra một ví dụ.

In [ ]:
for a,expected in [([],[]),([1],[1]),([1,None,2,3],[1,2,3]),([4,2,6,1,3,5,7],[4,2,1,3,6,5,7])]:
    assert solve(tree(a))==expected
    assert Solution().preorderTraversal(tree(a))==expected
    assert baseline(tree(a))==expected
print('PASS 4 ca preorder')

### 10.1. Kiểm thử cây nhỏ sinh ngẫu nhiên rồi đối chiếu recursion

Một bộ test mạnh có cấu trúc cây đa dạng hơn 3 ví dụ trong đề.
Ta sinh cây kích thước 0..35 nút; mỗi node có thể được nối trái hoặc phải.
Đồng thời có thể có **giá trị trùng** để phát hiện lỗi vô tình khử trùng.
Trên cây nhỏ, baseline recursion an toàn và đóng vai trò **oracle độc lập**:
`solve(root)` phải trùng kết quả `baseline(root)`.
Hãy sử dụng `random.Random(seed)` để kết quả tái hiện được khi có lỗi.
Bài kiểm thử không sửa cây thật và chỉ in số ca PASS, tránh output nghìn dòng.

In [ ]:
import random
rng=random.Random(144 if 'preorder' in solve.__name__ else 145)
def random_tree_for_test(n,rng):
    if n==0:return None
    nodes=[TreeNode(rng.randrange(-3,4)) for _ in range(n)]
    possible=[nodes[0]]
    for child in nodes[1:]:
        while True:
            parent=rng.choice(possible)
            options=[side for side in ('left','right') if getattr(parent,side) is None]
            if not options:
                possible.remove(parent)
                continue
            side=rng.choice(options);setattr(parent,side,child)
            possible.append(child)
            if parent.left is not None and parent.right is not None:possible.remove(parent)
            break
    return nodes[0]
for n in range(36):
    for _ in range(12):
        root=random_tree_for_test(n,rng)
        assert solve(root)==baseline(root)
print('PASS 432 cây ngẫu nhiên kích thước 0..35')

### 10.2. Cây dây rất sâu để kiểm iterative stack

Cách recursion có thể bị giới hạn bởi call stack Python;
trong khi explicit stack có thể xử lý cây dây vài nghìn nút.
Ta dựng cây chỉ có con trái với 2.000 node và chạy hàm `solve`.
**Không chạy baseline recursion ở trường hợp này** để tránh RecursionError,
chỉ kiểm output gồm đủ 2.000 phần tử và đúng hai đầu được suy ra bằng tay.
Test này chỉ nhằm cho thấy ưu điểm của giải pháp iterative khi h lớn;
memory explicit stack vẫn O(h), không phải O(1).

In [ ]:
chain_root=make_left_chain_demo(2000)
long_output=solve(chain_root)
assert len(long_output)==2000
print('cây dây 2000 node: số phần tử:',len(long_output),
      '5 phần tử đầu:',long_output[:5], '5 phần tử cuối:',long_output[-5:])

### 10.3. Kiểm thử thuộc tính output: đủ số node và giữ nguyên đa tập giá trị

Dù chọn preorder hay postorder, thuật toán phải ghi mỗi node đúng một lần;
vì vậy `len(output)` bằng số node thực, và `Counter(output)`
phải bằng tần suất `.val` của toàn bộ nút.
Kiểm thuộc tính này độc lập với thứ tự, sau đó tiếp tục dùng baseline
để xác nhận chính xác **thứ tự**. Một output có đủ số nhưng đảo thứ tự
vẫn sai đề, do đó cần cả hai lớp kiểm tra.
Nhấn mạnh thuật toán traversal **không khử trùng giá trị**.
Dùng vài cây có node cùng số, số âm và node rỗng.

In [ ]:
from collections import Counter
trees=[[],[5],[5,5,5],[1,2,3,None,4,2,2],[0,-1,1,-1,None,0,0]]
for values in trees:
    out=solve(tree(values))
    raw=[x for x in values if x is not None]
    assert Counter(out)==Counter(raw)
    assert len(out)==len(raw)
    assert out==baseline(tree(values))
print('PASS bảo toàn số node và đa tập giá trị')

## 11. Time / Space Complexity — có tính tới stack và output

Gọi `n` là số node thật và `h` là chiều cao cây. Duyệt **preorder** cần đi tới mỗi node đúng một lần: Time **O(n)**.
Hàm đệ quy baseline giữ call stack tới O(h); lời giải lặp giữ explicit stack tới O(h) công việc tại thời điểm cao nhất (tối đa O(n) nếu cây hình dây).
Danh sách **output luôn chứa n giá trị** nên riêng bộ nhớ trả kết quả cần O(n), tách khỏi extra working memory O(h).
Đối với cây đầy, h có thể O(log n); với cây lệch, h có thể O(n). Không được tuyên bố mọi cây đều cao O(log n).
Giải pháp iterative tránh RecursionError do giới hạn gọi hàm Python; nó không làm mất chi phí bộ nhớ lưu công việc.

## 12. Các lỗi người mới hay mắc

- ❌ Push trái trước phải: pop sẽ xử lý **phải trước**.
- ❌ Ghi node sau khi xét con: thành postorder.
- ❌ Quên cây rỗng phải trả `[]`.

Khi debug: cố tạo input nhỏ nhất gây sai, trace biến ở **ngay trước lệnh `if`/`return`**, rồi sửa nguyên nhân thay vì thêm điều kiện tùy tiện.

In [ ]:
root_bug144=tree([1,2,3])
print('cây có hai con:',show_tree(root_bug144),'đúng preorder:',solve(root_bug144))
assert solve(root_bug144)==[1,2,3]

### 12.1. Sai lầm: đẩy trái trước phải làm đảo hai nhánh

Stack LIFO khác queue FIFO. Nếu `append(left)` rồi `append(right)`,
lần `pop()` kế tiếp sẽ là **right**, khiến traversal thành Root–Right–Left.
Sai lầm này có thể không lộ trên cây chỉ một nhánh.
Hãy chọn cây nhỏ nhất có hai con `[1,2,3]`:
preorder đúng `[1,2,3]`, bản push sai `[1,3,2]`.
So sánh hai bản và rút ra quy tắc push **ngược** thứ tự muốn pop.
Đừng sửa bằng cách đảo output toàn cục: đảo Root–Right–Left không cho preorder đúng nói chung.

In [ ]:
def wrong_push_order144(root):
    if root is None:return []
    stack=[root];out=[]
    while stack:
        node=stack.pop();out.append(node.val)
        if node.left:stack.append(node.left)
        if node.right:stack.append(node.right)
    return out
r=tree([1,2,3])
print('sai:',wrong_push_order144(r),'đúng:',solve(r))
assert wrong_push_order144(r)==[1,3,2] and solve(r)==[1,2,3]

### 12.2. Sai lầm: ghi sau khi xong hai con thì không còn là preorder

Preorder định nghĩa rõ Root–Left–Right. Nếu chờ xử lý hai con rồi mới ghi,
bạn đang tạo postorder (Left–Right–Root).
Trên cây `[2,1,3]`, preorder đúng là `[2,1,3]`, postorder `[1,3,2]`.
Một số test chỉ gồm một node sẽ không phân biệt hai cách vì kết quả giống nhau.
Vì thế ma trận test phải có cây 3 node với cả trái và phải,
và ít nhất một cây nhiều tầng. Khi debug, tìm **thời điểm append `.val`**,
không chỉ kiểm `left/right` có đúng không.

In [ ]:
def postorder_for_contrast144(root):
    if root is None:return []
    return postorder_for_contrast144(root.left)+postorder_for_contrast144(root.right)+[root.val]
test_root=tree([2,1,3])
print('ghi sau hai con:',postorder_for_contrast144(test_root))
print('preorder đúng:',solve(test_root))
assert postorder_for_contrast144(test_root)!=solve(test_root)

## 13. Nhận diện dạng bài và vận dụng

Với DFS và một thứ tự ghi node cụ thể, đặt câu hỏi: **ghi ở thời điểm nào** và stack LIFO sắp xếp lệnh push ra sao?

**Chuyển giao:** hãy tìm một bài khác có cùng dấu hiệu đầu vào và tập trung vào lý do giữ bất biến, không học thuộc tên biến.

## 14. Biến thể có ý nghĩa

So với inorder của bài #94: **chỉ thay thời điểm ghi node** đã thay toàn bộ kết quả.

Biến thể này dùng lại kỹ thuật vừa học, nhưng có một thay đổi nhỏ giúp kiểm tra bạn đã hiểu hay chưa.

In [ ]:
r=tree([4,2,6,1,3,5,7]);print('preorder=',solve(r),'inorder=',[1,2,3,4,5,6,7])

### 14.1. Biến thể: dùng stack để trả danh sách node thực, không chỉ value

Đề #144 yêu cầu danh sách giá trị, nhưng nhiều bài cây khác
cần giữ chính object node để sửa hoặc đánh dấu.
Ta có thể thay `result.append(node.val)` bằng `result.append(node)`.
Thứ tự thăm không đổi, nhưng kiểu output đổi từ `list[int]` sang `list[TreeNode]`.
Cẩn thận: hai node có cùng `.val` vẫn phải xuất hiện riêng biệt.
Đây là bài luyện hiểu sự khác nhau giữa **tham chiếu** và **giá trị**.
Không nộp phiên bản node này cho bài #144.

In [ ]:
def preorder_nodes144(root):
    if root is None:return []
    stack=[root];out=[]
    while stack:
        node=stack.pop();out.append(node)
        if node.right:stack.append(node.right)
        if node.left:stack.append(node.left)
    return out
r=tree([1,2,2])
nodes=preorder_nodes144(r)
print('nodes:',nodes,'giá trị:',[n.val for n in nodes])
assert nodes[1] is not nodes[2] and [n.val for n in nodes]==solve(r)

### 14.2. Biến thể: đếm số node và tổng giá trị trong khi preorder

Một khi đã biết duyệt thăm mỗi nút đúng một lần, ta có thể
đổi thao tác `append` thành **cập nhật một biến đếm** hoặc **tích lũy tổng**.
Đếm số node dùng `count += 1`; tổng các `.val` dùng `total += node.val`.
Cả hai vẫn O(n) time; riêng đếm/tổng không cần output list O(n),
nhưng stack DFS vẫn có O(h) memory ở cây cao h.
Đừng nhầm phép gộp này là lời giải đề #144: đề vẫn yêu cầu đúng thứ tự list số.
Đây là kỹ năng chuyển giao từ traversal sang xử lý cây.

In [ ]:
def sum_preorder144(root):
    if root is None:return 0,0
    stack=[root];count=0;total=0
    while stack:
        node=stack.pop();count+=1;total+=node.val
        if node.right:stack.append(node.right)
        if node.left:stack.append(node.left)
    return count,total
for arr in ([4,2,6,1,3,5,7],[],[2,-1,-1]):
    print(arr,'số nút, tổng:',sum_preorder144(tree(arr)))
assert sum_preorder144(tree([2,-1,-1]))==(3,0)

## 15. Hệ thống luyện tập — 6 bài từ cơ bản đến nâng cao

**Bài 1 — Dự đoán output Python:** Cho stack rỗng, push `"right"`, push `"left"` rồi pop 2 lần. Dự đoán thứ tự pop.

**Bài 2 — Trace tay:** Trace cây `[4,2,6,1,3,5,7]`: ghi stack trước/sau và kết quả sau mỗi pop; chú ý push phải trước trái.

**Bài 3 — Hoàn thiện code chính:** tự viết hàm `exercise_preorder144(root)` mà không nhìn cell nộp.

**Bài 4 — Biến thể:** Viết hàm trả `list[TreeNode]` chứa chính các node theo preorder (không phải chỉ các `.val`).

**Bài 5 — Viết test phản ví dụ:** Tự tạo 5 bộ test `(level_order, expected)` gồm rỗng, một nút, cây dây, cây hai nhánh, node trùng giá trị.

**Bài 6 — Mini challenge:** Biến thể: trả `list[tuple[int,int]]` gồm `(node.val, depth)` theo preorder; gốc có độ sâu 1.

**Cách tự học:** làm bài 1–2 trên giấy → viết TODO bài 3–5 → tự kiểm bằng test của chính mình → thử bài 6. Hãy ghi rõ lý do chọn trạng thái, ranh giới vòng lặp và điều kiện dừng; **đáp án tham khảo nằm sau toàn bộ đề bài**.

### Bài 3 — TODO: tự viết lại Binary Tree Preorder Traversal

Viết code vào hàm ở cell tiếp theo theo đúng kiểu input/output của đề.
Không gọi hàm TODO khi chạy toàn bộ notebook. Khi hoàn thành, tự bỏ `raise NotImplementedError` rồi chạy test được gợi ý dưới hàm.
**Tiêu chí tự chấm:** giải thích được vì sao code đúng ngay cả ở trường hợp nhỏ nhất, và không làm thao tác dư thừa so với lời giải chính.

In [ ]:
def exercise_preorder144(root):
    # TODO: nếu root rỗng trả []
    # TODO: stack giữ node chưa xử lý, pop -> append(node.val)
    # TODO: push right trước left để lần pop sau xử lý left trước
    raise NotImplementedError("Tự viết trước khi kiểm thử")
# assert exercise_preorder144(tree([1,None,2,3]))==[1,2,3]

### Bài 4 — TODO: đổi yêu cầu, giữ kỹ thuật cốt lõi

Viết hàm trả `list[TreeNode]` chứa chính các node theo preorder (không phải chỉ các `.val`).

Đừng chép y nguyên code submit: hãy chỉ ra đầu ra thay đổi ở đâu; tự tạo 2 input mới để kiểm thử.

In [ ]:
def exercise_preorder_nodes144(root):
    # TODO: dùng stack giống preorder, nhưng lưu node object vào list output.
    raise NotImplementedError("Chưa làm bài 4")
# r=tree([1,2,3]); assert exercise_preorder_nodes144(r)[0] is r

### Bài 5 — TODO: tạo bộ kiểm thử có chủ đích

Tự tạo 5 bộ test `(level_order, expected)` gồm rỗng, một nút, cây dây, cây hai nhánh, node trùng giá trị.

Mỗi test phải có **expected tự tính trước**, không được lấy `expected=solve(input)`. Một test tốt thường có mục tiêu bắt một kiểu sai cụ thể. Nếu có node, dựng dữ liệu mới ở mỗi lượt test.

In [ ]:
def exercise_cases144():
    # TODO: trả [(level_order_list, expected_preorder_list), ...], tối thiểu 5 ca.
    raise NotImplementedError("Chưa làm bài 5")

### Bài 6 — Mini challenge chuyển giao kỹ thuật (khó)

**Yêu cầu:** Biến thể: trả `list[tuple[int,int]]` gồm `(node.val, depth)` theo preorder; gốc có độ sâu 1.

    Bài 6 không thuộc đề gốc. Hãy tự phân tích **miền input, output, tính đúng**
    và chọn một phản ví dụ nhỏ trước khi code. Nếu thay đổi cấu trúc lời giải,
    cần nói rõ điều kiện nào đã thay đổi so với bài gốc.
    Không gọi hàm có TODO trong lần chạy toàn bộ notebook: để nguyên `NotImplementedError`
    cho tới khi bạn tự điền mã và tự bật test.
    **Kiểm tra:** so kết quả với đáp án cuối, nhưng chỉ sau khi đã tự làm.

In [ ]:
def exercise_preorder_with_depth144(root):
    # TODO: stack chứa (node, depth)
    # TODO: lấy từng cặp, append (node.val,depth); push right/left với depth+1
    raise NotImplementedError("Mini challenge chưa giải")
# assert exercise_preorder_with_depth144(tree([1,2,3]))==[(1,1),(2,2),(3,2)]

## 16. Gợi ý theo ba cấp — chỉ đọc sau khi tự thử

**Cấp 1:** preorder ghi gốc rồi mới xử lý trái và phải.
**Cấp 2:** stack `append(right)` trước `append(left)`; pop cuối trước.
**Cấp 3:** Bài 4 giữ nguyên thuật toán chỉ đổi `append(node.val)` thành `append(node)`; bài 6 thêm depth vào mỗi công việc.
**Nâng cao:** gốc depth 1, mỗi lần đi xuống một cạnh mới tăng depth 1. Vẫn duyệt được khi root None.

**Cách sử dụng gợi ý:** bắt đầu ở cấp 1; nếu vẫn chưa ra, chỉ mở cấp 2 rồi tự viết tiếp. Không đọc lời giải đầy đủ ngay khi vừa gặp khó. Với bài 6, thử tự xác định input nhỏ nhất gây sai trong phương án bạn đề xuất.

## 17. Đáp án tham khảo — chỉ đọc sau khi đã làm TODO

Các hàm **đáp án** dùng tên khác với `exercise_*` để không vô tình thực thi code chưa hoàn thiện.
Đáp án bài 3 được định nghĩa trước, rồi đến các biến thể bài 4–6 và bộ test tương ứng.
Sau khi chạy, hãy so sánh logic từng dòng với lời giải chính và tìm lý do các cài đặt tương đương, không chỉ so con số output.

In [ ]:
def exercise_core_answer(root):
    if root is None:return []
    stack=[root];ans=[]
    while stack:
        node=stack.pop()
        ans.append(node.val)
        if node.right is not None:stack.append(node.right)
        if node.left is not None:stack.append(node.left)
    return ans

print("Đã định nghĩa đáp án tham khảo; xem lại từng dòng rồi tự gõ lại.")

### Đáp án bài 4 — biến thể với yêu cầu đầu ra mới

Bài 4 yêu cầu: Viết hàm trả `list[TreeNode]` chứa chính các node theo preorder (không phải chỉ các `.val`).

    So với lời giải gốc, hãy chỉ ra phần nào giữ nguyên:
    cú pháp duyệt, trạng thái trước/sau, bất biến. Phần thay đổi phải tương ứng
    đúng với **đầu ra mới**, không làm thay chữ ký code submit #
    hoặc vô tình lẫn thêm điều kiện ngoài phạm vi bài gốc.
    Đoạn đáp án dưới đây có assert riêng để kiểm tính đúng.

In [ ]:
def answer_preorder_nodes144(root):
    if root is None:return []
    stack=[root];result=[]
    while stack:
        node=stack.pop();result.append(node)
        if node.right:stack.append(node.right)
        if node.left:stack.append(node.left)
    return result
r=tree([1,2,2])
assert answer_preorder_nodes144(r)[0] is r
assert [node.val for node in answer_preorder_nodes144(r)]==[1,2,2]
print('PASS đáp án bài 4 — trả node tham chiếu')

### Đáp án bài 6 và kiểm thử đáp án bài 5

**Bài 6:** Biến thể: trả `list[tuple[int,int]]` gồm `(node.val, depth)` theo preorder; gốc có độ sâu 1.

    Đọc suy luận trước khi so code: giữ trạng thái nào; tại mỗi bước
    trạng thái cập nhật ra sao; vì sao khi dừng, kết quả đáp ứng yêu cầu.
    Bài 5 là tự xây bộ test: đáp án tham khảo còn có các mẫu độc lập,
    chạy kèm assert nhưng **không hề chạy TODO của người học**.
    Nếu bộ test bạn tự nghĩ khác, điều đó bình thường: hãy đảm bảo cả
    test của bạn và test tham khảo đều có expected đúng.

In [ ]:
def answer_preorder_with_depth144(root):
    if root is None:return []
    stack=[(root,1)];out=[]
    while stack:
        node,depth=stack.pop();out.append((node.val,depth))
        if node.right:stack.append((node.right,depth+1))
        if node.left:stack.append((node.left,depth+1))
    return out
for arr,expected in [([],[]),([1],[(1,1)]),([1,2,3],[(1,1),(2,2),(3,2)]),([1,None,2,3],[(1,1),(2,2),(3,3)])]:
    assert answer_preorder_with_depth144(tree(arr))==expected
print('PASS đáp án mini challenge — preorder và depth')

cases144=[([],[]),([1],[1]),([1,2,3],[1,2,3]),([1,None,2,3],[1,2,3]),([5,5,5],[5,5,5]),([4,2,6,1,3,5,7],[4,2,1,3,6,5,7])]
for values,expected in cases144:assert solve(tree(values))==expected
print('PASS 6 test tham khảo preorder')

### Lời giải thích và đáp án ngắn cho cả 6 bài

**Bài 1:** do LIFO, `pop()` trả `"left"`, rồi `"right"`.
**Bài 2:** cây `[4,2,6,1,3,5,7]` cho `[4,2,1,3,6,5,7]` khi push phải trước trái.
**Bài 3:** xem `exercise_core_answer`, mỗi node thật được append đúng một lần.
**Bài 4:** `answer_preorder_nodes144` giữ chính object TreeNode; hai node có cùng val không bị gộp.
**Bài 5:** bộ test còn kiểm cây có cả hai nhánh để phát hiện lỗi push đảo thứ tự.
**Bài 6:** `answer_preorder_with_depth144`: stack `(node,depth)`, gốc depth 1.

Đối với bài tập viết code, hãy so sánh cả **điều kiện lặp**, **thứ tự cập nhật** và **cách return** chứ không chỉ so giá trị cuối. Các hàm `answer_*` của bài 4 và 6 nằm ở những cell ngay phía trên để người học có thể chạy riêng sau khi tự làm.

## 18. Tóm tắt và checklist trước khi nộp

**Tư duy phải tự diễn đạt được bằng lời:** nêu bài toán bằng một câu, chỉ rõ trạng thái nào đủ để quyết định bước kế tiếp, rồi mới viết code. Đọc lại các ví dụ khó trong notebook; khi một điều kiện `if` làm bạn phân vân, hãy lập bảng giá trị **ngay trước và sau** câu lệnh ấy.

1. Tôi đã phân biệt đầu vào/đầu ra, và đọc đúng chữ ký `class Solution` của bài.
2. Tôi đã tự chạy thử ví dụ nhỏ, biên, phản ví dụ và kiểm thử đối chiếu nhiều cách giải.
3. Tôi hiểu từng cấu trúc Python, có thể giải thích vì sao dùng nó thay vì cách khác.
4. Tôi chỉ ra được bất biến: sau bước thứ k, điều gì **luôn đúng** và tại sao?
5. Tôi phân tích Time và Space dựa trên số lượng phần tử/nút/bít và có tính tới output/stack/chuỗi tạm.
6. Tôi đã hoàn thành TODO **trước** khi kéo tới đáp án tham khảo.
7. Tôi chỉ sao chép `class Solution` để nộp LeetCode; các hàm dựng input, trace và bộ test dành cho notebook, không dành cho submit.

**Tự đánh giá:** bạn hiểu bài khi tự viết được lời giải trên một trang trống và xử lý được một biến thể mà không sao chép mã cũ.

**Mẫu nhận dạng của bài 144:** DFS ghi Root trước Left/Right → explicit stack, push right rồi left, output preorder.